In [7]:
import pandas as pd
import numpy as np

In [8]:
df=pd.read_csv("/content/drive/MyDrive/Dataset/public_transport_delays.csv")

In [9]:
print(df.head())

  trip_id        date      time transport_type  route_id origin_station  \
0  T00000  2023-01-01  05:00:00           Tram  Route_15     Station_31   
1  T00001  2023-01-01  05:15:00          Metro  Route_12     Station_49   
2  T00002  2023-01-01  05:30:00            Bus  Route_16     Station_29   
3  T00003  2023-01-01  05:45:00           Tram  Route_19     Station_26   
4  T00004  2023-01-01  06:00:00           Tram   Route_8     Station_18   

  destination_station scheduled_departure scheduled_arrival  \
0           Station_6            05:02:00          05:55:00   
1          Station_32            05:16:00          05:55:00   
2          Station_42            05:33:00          06:17:00   
3          Station_18            05:49:00          06:08:00   
4          Station_15            06:00:00          06:35:00   

   actual_departure_delay_min  ...  wind_speed_kmh precipitation_mm  \
0                          12  ...              46             13.0   
1                          1

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   trip_id                     2000 non-null   object 
 1   date                        2000 non-null   object 
 2   time                        2000 non-null   object 
 3   transport_type              2000 non-null   object 
 4   route_id                    2000 non-null   object 
 5   origin_station              2000 non-null   object 
 6   destination_station         2000 non-null   object 
 7   scheduled_departure         2000 non-null   object 
 8   scheduled_arrival           2000 non-null   object 
 9   actual_departure_delay_min  2000 non-null   int64  
 10  actual_arrival_delay_min    2000 non-null   int64  
 11  weather_condition           2000 non-null   object 
 12  temperature_C               2000 non-null   float64
 13  humidity_percent            2000 

In [11]:
print(df.isnull().sum())

trip_id                          0
date                             0
time                             0
transport_type                   0
route_id                         0
origin_station                   0
destination_station              0
scheduled_departure              0
scheduled_arrival                0
actual_departure_delay_min       0
actual_arrival_delay_min         0
weather_condition                0
temperature_C                    0
humidity_percent                 0
wind_speed_kmh                   0
precipitation_mm                 0
event_type                    1173
event_attendance_est             0
traffic_congestion_index         0
holiday                          0
peak_hour                        0
weekday                          0
season                           0
delayed                          0
dtype: int64


In [12]:
df['event_type']=df['event_type'].fillna('No Event')

In [14]:
from sklearn.preprocessing import LabelEncoder

In [15]:
le=LabelEncoder()

In [17]:
categorical_cols=['trip_id','date','time','transport_type','route_id','origin_station','destination_station','scheduled_departure','scheduled_arrival','weather_condition','event_type','season']
for col in categorical_cols:df[col]=le.fit_transform(df[col])

In [18]:
print(df.head())

   trip_id  date  time  transport_type  route_id  origin_station  \
0        0     0    20               3         6              24   
1        1     0    21               1         3              43   
2        2     0    22               0         7              21   
3        3     0    23               3        10              18   
4        4     0    24               3        18               9   

   destination_station  scheduled_departure  scheduled_arrival  \
0                   46                   99                264   
1                   25                  103                264   
2                   36                  110                278   
3                    9                  116                271   
4                    6                  117                291   

   actual_departure_delay_min  ...  wind_speed_kmh  precipitation_mm  \
0                          12  ...              46              13.0   
1                          15  ...              11

In [19]:
x=df.drop('delayed',axis=1)
y=df['delayed']

In [20]:
from sklearn.model_selection import train_test_split

In [21]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [22]:
pip install xgboost

In [23]:
from xgboost import XGBClassifier

In [24]:
model=XGBClassifier()

In [25]:
model.fit(x_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [26]:
y_pred=model.predict(x_test)

In [27]:
from sklearn.metrics import accuracy_score

In [28]:
accuracy=accuracy_score(y_test,y_pred)
print("Accuracy:",accuracy*100)

Accuracy: 100.0


In [29]:
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(y_test,y_pred)
print(cm)

[[106   0]
 [  0 294]]


In [30]:
from sklearn.metrics import classification_report
cr=classification_report(y_test,y_pred)
print(cr)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       106
           1       1.00      1.00      1.00       294

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400



In [32]:
import pickle
pickle.dump(model,open("bus_delay_model.pkl","wb"))
print("Model saved successfully")

Model saved successfully


In [33]:
sample=x.iloc[[0]]
prediction=model.predict(sample)
if prediction[0]==1:
  print("Prediction: Delayed")
else:
  print("Prediction: Not Delayed")

Prediction: Not Delayed
